# Stage 3: all available models x S0, S2 on all four test sets

Bedrock models run in-process; open models are served one at a time through `scripts/serve.sh`. The large
reference model runs S2 only. At the end the top 3 models (S2, mean over the four sets) are written to
`results/stage3_ranking.json`.

In [ ]:
from pathlib import Path
import json, os, subprocess
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
from t2sbench.bench import load_results
from t2sbench.report import aggregate
pd.set_option("display.max_colwidth", 120)
RUN = False  # set True to execute the stage from here (it can take hours and cost money)

def sh(cmd):
    print("$", cmd)
    if RUN:
        subprocess.run(cmd, shell=True, check=True)

def summary(stage):
    df = load_results(Path("results"), stage)
    if df.empty:
        print(f"no results for {stage} yet")
        return df
    agg = aggregate(df)
    return agg[["model", "adapter", "strategy", "dataset", "n", "ex", "ci_lo", "ci_hi", "syntax_ok", "no_answer",
                "time_median", "time_p90", "in_tokens_mean", "out_tokens_mean", "cost_per_100"]].sort_values(
                ["dataset", "ex"], ascending=[True, False])

In [ ]:
sh("uv run t2sbench stage 3")

In [ ]:
summary("stage3")

In [ ]:
r = Path("results/stage3_ranking.json")
if r.exists():
    print(json.loads(r.read_text()))

In [ ]:
if Path("logs/skipped.jsonl").exists():
    display(pd.read_json("logs/skipped.jsonl", lines=True))